# Model Benchmarks: All Models

Compare all predictive models of human role choice (aggregate-tuned and switch-stage-tuned).

**Tuned Models (4):**
- **Bayesian Walk**: Stick-or-switch with softmax best response
- **Bayesian Threshold**: Rational stick-or-switch (value gap must exceed delta)
- **Bayesian-Value**: Softmax over expected values given posterior beliefs
- **Random Walk**: Fixed probability of switching to a random different role

**Baselines (7):**
- Random, Optimal, Top-7, Random-to-Optimal, Copy Others, Contradict Others

**Data:** March 6 + March 18 exports (human rounds only).

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    softmax_role_dist, combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds and teacher_forced_predictions
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

## 1. Load Data

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## 2. Load Tuned Parameters

In [3]:
SCRIPT_DIR = Path('.')

with open(SCRIPT_DIR / 'bayesian_walk' / 'bayesian_walk_params.json') as f:
    bw_params = json.load(f)
with open(SCRIPT_DIR / 'bayesian_value' / 'bayesian_value_params.json') as f:
    bv_params = json.load(f)
with open(SCRIPT_DIR / 'bayesian_thresh' / 'bayesian_thresh_params.json') as f:
    bt_params = json.load(f)
with open(SCRIPT_DIR / 'random_walk' / 'random_walk_params.json') as f:
    rw_params = json.load(f)
print('Loaded tuned parameters:')
for name, p in [('Bayesian Walk', bw_params), ('Bayesian Threshold', bt_params),
                ('Bayesian-Value', bv_params), ('Random Walk', rw_params)]:
    agg = p['aggregate_tuned']
    sw = p['switch_stage_tuned']
    agg_r = agg.get('combo_r', agg.get('bayesian_thresh', agg).get('combo_r', '?'))
    sw_r = sw.get('combo_r', sw.get('bayesian_thresh', sw).get('combo_r', '?'))
    print(f'  {name}: aggregate combo_r={agg_r:.4f}, switch combo_r={sw_r:.4f}')

Loaded tuned parameters:
  Bayesian Walk: aggregate combo_r=0.5407, switch combo_r=0.2971
  Bayesian Threshold: aggregate combo_r=0.4744, switch combo_r=0.2493
  Bayesian-Value: aggregate combo_r=0.4022, switch combo_r=0.2865
  Random Walk: aggregate combo_r=0.4510, switch combo_r=0.2053


## 3. Helper Functions

In [4]:
def run_baseline(records, predict_fn):
    """Wrap a stage-level baseline predict_fn into the format run_predictions expects."""
    def wrapped(record):
        preds = []
        for s, human_combo in enumerate(record['stage_roles']):
            prev = record['stage_roles'][s - 1] if s > 0 else None
            dist = predict_fn(record, s, prev, record['env_config'])
            marg = np.zeros(3)
            for combo, prob in dist.items():
                for c in combo:
                    marg[ROLE_CHAR_TO_IDX[c]] += prob
            marg /= 3.0
            preds.append({
                'predicted_dist': dist,
                'human_combo': human_combo,
                'model_marginal': marg,
            })
        return preds
    return run_predictions(records, wrapped)


def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered

## 4. Define Models

In [5]:
# --- Bayesian Walk ---
def make_bayesian_walk(tau_prior, tau_softmax, epsilon, epsilon_switch):
    def predict_fn(record):
        env = record['env_config']
        values = env['values']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        prev_roles = None

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break
            intent = record['lds'][turn_idx]
            thp = int(min(max(0, team_hp), team_max_hp))
            ehp = int(min(max(0, enemy_hp), enemy_max_hp))

            switch_dist = [softmax_role_dist(i, intent, thp, ehp, prior, values, tau_softmax)
                           for i in range(3)]
            per_agent = []
            for i in range(3):
                if prev_roles is None:
                    per_agent.append(switch_dist[i])
                else:
                    stick = np.zeros(3)
                    stick[prev_roles[i]] = 1.0
                    mixed = (1.0 - epsilon_switch) * stick + epsilon_switch * switch_dist[i]
                    per_agent.append(mixed)

            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = human_roles
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn


# --- Bayesian Threshold ---
def expected_values_per_role(agent_i, intent, team_hp, enemy_hp, prior, values):
    other_agents = [a for a in range(3) if a != agent_i]
    other_probs = np.sum(prior, axis=agent_i)
    total = other_probs.sum()
    other_probs = other_probs / total if total > 0 else np.ones((3, 3)) / 9.0

    ev = np.zeros(3)
    for r_i in range(3):
        for r_j in range(3):
            for r_k in range(3):
                roles = [0, 0, 0]
                roles[agent_i] = r_i
                roles[other_agents[0]] = r_j
                roles[other_agents[1]] = r_k
                flat_idx = roles[0] * 9 + roles[1] * 3 + roles[2]
                ev[r_i] += other_probs[r_j, r_k] * float(values[flat_idx, intent, team_hp, enemy_hp])
    return ev


def threshold_role_dist(agent_i, intent, team_hp, enemy_hp, prior, values,
                        current_role, delta, tau):
    ev = expected_values_per_role(agent_i, intent, team_hp, enemy_hp, prior, values)
    current_val = ev[current_role]
    candidates = [r for r in range(3) if r != current_role and (ev[r] - current_val) > delta]

    if not candidates:
        dist = np.zeros(3)
        dist[current_role] = 1.0
        return dist

    candidate_vals = np.array([ev[r] for r in candidates])
    scaled = candidate_vals / tau
    scaled -= scaled.max()
    exp_vals = np.exp(scaled)
    probs = exp_vals / exp_vals.sum()

    dist = np.zeros(3)
    for i, r in enumerate(candidates):
        dist[r] = probs[i]
    return dist


def make_bayesian_thresh(tau_prior, tau_softmax, epsilon, delta):
    def predict_fn(record):
        env = record['env_config']
        values = env['values']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        prev_roles = None

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break
            intent = record['lds'][turn_idx]
            thp = int(min(max(0, team_hp), team_max_hp))
            ehp = int(min(max(0, enemy_hp), enemy_max_hp))

            per_agent = []
            for i in range(3):
                if prev_roles is None:
                    per_agent.append(softmax_role_dist(i, intent, thp, ehp, prior, values, tau_softmax))
                else:
                    per_agent.append(threshold_role_dist(
                        i, intent, thp, ehp, prior, values,
                        current_role=prev_roles[i], delta=delta, tau=tau_softmax))

            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = human_roles
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn


# --- Random Walk ---
def make_random_walk(eps):
    def predict_fn(record):
        preds = []
        for s, human_combo in enumerate(record['stage_roles']):
            prev = record['stage_roles'][s - 1] if s > 0 else None
            if prev is None:
                dist = {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
            else:
                dist = {}
                for combo in ALL_ROLE_COMBOS:
                    p = 1.0
                    for i, (c, prev_c) in enumerate(zip(combo, prev)):
                        p *= (1.0 - eps) if c == prev_c else (eps / 2.0)
                    dist[combo] = p
                total = sum(dist.values())
                dist = {c: p / total for c, p in dist.items()}
            marg = np.zeros(3)
            for combo, prob in dist.items():
                for c in combo:
                    marg[ROLE_CHAR_TO_IDX[c]] += prob
            marg /= 3.0
            preds.append({'predicted_dist': dist, 'human_combo': human_combo, 'model_marginal': marg})
        return preds
    return predict_fn




In [6]:
# --- Simple Baselines ---

def predict_random(record, stage_idx, prev_combo, env_config):
    return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}


def predict_optimal(record, stage_idx, prev_combo, env_config):
    values = env_config['values']
    lds = record['lds']
    turn_idx = stage_idx * TURNS_PER_STAGE
    intent = lds[turn_idx] if turn_idx < len(lds) else 0
    team_hp = int(env_config['team_max_hp'])
    enemy_hp = int(env_config['enemy_max_hp'])

    vals = []
    for combo in ALL_ROLE_COMBOS:
        idx = ROLE_CHAR_TO_IDX[combo[0]] * 9 + ROLE_CHAR_TO_IDX[combo[1]] * 3 + ROLE_CHAR_TO_IDX[combo[2]]
        vals.append(float(values[idx, intent, min(team_hp, values.shape[2]-1), min(enemy_hp, values.shape[3]-1)]))
    max_val = max(vals)
    optimal_combos = [c for c, v in zip(ALL_ROLE_COMBOS, vals) if abs(v - max_val) < 1e-8]
    dist = {c: 0.0 for c in ALL_ROLE_COMBOS}
    for c in optimal_combos:
        dist[c] = 1.0 / len(optimal_combos)
    return dist


def make_topk(k=7):
    def predict(record, stage_idx, prev_combo, env_config):
        values = env_config['values']
        lds = record['lds']
        turn_idx = stage_idx * TURNS_PER_STAGE
        intent = lds[turn_idx] if turn_idx < len(lds) else 0
        team_hp = int(env_config['team_max_hp'])
        enemy_hp = int(env_config['enemy_max_hp'])

        combo_vals = []
        for combo in ALL_ROLE_COMBOS:
            idx = ROLE_CHAR_TO_IDX[combo[0]] * 9 + ROLE_CHAR_TO_IDX[combo[1]] * 3 + ROLE_CHAR_TO_IDX[combo[2]]
            combo_vals.append((combo, float(values[idx, intent, min(team_hp, values.shape[2]-1), min(enemy_hp, values.shape[3]-1)])))
        combo_vals.sort(key=lambda x: -x[1])
        top = combo_vals[:k]
        dist = {c: 0.0 for c in ALL_ROLE_COMBOS}
        for c, v in top:
            dist[c] = 1.0 / k
        return dist
    return predict


def predict_random_to_optimal(record, stage_idx, prev_combo, env_config):
    alpha = stage_idx / max(len(record['stage_roles']) - 1, 1)
    random_dist = {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
    optimal_dist = predict_optimal(record, stage_idx, prev_combo, env_config)
    return {c: (1 - alpha) * random_dist[c] + alpha * optimal_dist[c] for c in ALL_ROLE_COMBOS}


def predict_copy_others(record, stage_idx, prev_combo, env_config):
    if prev_combo is None:
        return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
    dist = {}
    for combo in ALL_ROLE_COMBOS:
        p = 1.0
        for i in range(3):
            others = [prev_combo[j] for j in range(3) if j != i]
            if combo[i] in others:
                p *= 0.5
            else:
                p *= 0.0 if combo[i] not in others else 0.5
        dist[combo] = p
    total = sum(dist.values())
    if total > 0:
        return {c: p / total for c, p in dist.items()}
    return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}


def predict_contradict(record, stage_idx, prev_combo, env_config):
    if prev_combo is None:
        return {c: 1.0 / 27 for c in ALL_ROLE_COMBOS}
    dist = {}
    for combo in ALL_ROLE_COMBOS:
        p = 1.0
        for i in range(3):
            others = set(prev_combo[j] for j in range(3) if j != i)
            if combo[i] not in others:
                p *= 1.0
            else:
                p *= 0.1
        dist[combo] = p
    total = sum(dist.values())
    return {c: p / total for c, p in dist.items()}

## 5. Run All Models (Aggregate-Tuned)

In [7]:
all_models = []

# --- Bayesian Walk (aggregate-tuned) ---
p = bw_params['aggregate_tuned']
print('Running Bayesian Walk...')
print(f"  params: tau_prior={p['tau_prior']:.4f}, tau_softmax={p['tau_softmax']:.4f}, epsilon={p['epsilon']:.4f}, epsilon_switch={p['epsilon_switch']:.4f}")
bw_results = run_predictions(records, make_bayesian_walk(
    p['tau_prior'], p['tau_softmax'], p['epsilon'], p['epsilon_switch']))
bw_corrs = compute_pearson(bw_results)
bw_ll = compute_log_likelihood(bw_results)
bw_metrics = extract_metrics(bw_corrs)
bw_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in bw_ll.values()]))
bw_metrics['n_params'] = 4
all_models.append(('Bayesian Walk', bw_metrics, bw_corrs, bw_results))
print(f"  combo_r={bw_metrics['combo_r']:.4f}, marg_r={bw_metrics['marg_r']:.4f}")

# --- Bayesian Threshold (aggregate-tuned) ---
p = bt_params['aggregate_tuned']
if 'bayesian_thresh' in p:
    p = p['bayesian_thresh']
print('Running Bayesian Threshold...')
print(f"  params: tau_prior={p['tau_prior']:.4f}, tau_softmax={p['tau_softmax']:.4f}, epsilon={p['epsilon']:.4f}, delta={p['delta']:.4f}")
bt_results = run_predictions(records, make_bayesian_thresh(
    p['tau_prior'], p['tau_softmax'], p['epsilon'], p['delta']))
bt_corrs = compute_pearson(bt_results)
bt_ll = compute_log_likelihood(bt_results)
bt_metrics = extract_metrics(bt_corrs)
bt_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in bt_ll.values()]))
bt_metrics['n_params'] = 4
all_models.append(('Bayesian Threshold', bt_metrics, bt_corrs, bt_results))
print(f"  combo_r={bt_metrics['combo_r']:.4f}, marg_r={bt_metrics['marg_r']:.4f}")

# --- Bayesian-Value (aggregate-tuned) ---
p = bv_params['aggregate_tuned']
print('Running Bayesian-Value...')
print(f"  params: tau_prior={p['tau_prior']:.4f}, tau_softmax={p['tau_softmax']:.4f}, epsilon={p['epsilon']:.4f}")
bv_results = oms.run_all_predictions(records, tau_prior=p['tau_prior'],
                                      tau_softmax=p['tau_softmax'], epsilon=p['epsilon'])
bv_corrs = compute_pearson(bv_results)
bv_ll = compute_log_likelihood(bv_results)
bv_metrics = extract_metrics(bv_corrs)
bv_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in bv_ll.values()]))
bv_metrics['n_params'] = 3
all_models.append(('Bayesian-Value', bv_metrics, bv_corrs, bv_results))
print(f"  combo_r={bv_metrics['combo_r']:.4f}, marg_r={bv_metrics['marg_r']:.4f}")

# --- Random Walk (aggregate-tuned) ---
p = rw_params['aggregate_tuned']
print('Running Random Walk...')
print(f"  params: eps={p['eps']:.4f}")
rw_results = run_predictions(records, make_random_walk(p['eps']))
rw_corrs = compute_pearson(rw_results)
rw_ll = compute_log_likelihood(rw_results)
rw_metrics = extract_metrics(rw_corrs)
rw_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in rw_ll.values()]))
rw_metrics['n_params'] = 1
all_models.append(('Random Walk', rw_metrics, rw_corrs, rw_results))
print(f"  combo_r={rw_metrics['combo_r']:.4f}, marg_r={rw_metrics['marg_r']:.4f}")

# --- Simple Baselines ---
baselines = [
    ('Random', predict_random, 0),
    ('Optimal', predict_optimal, 0),
    ('Top-7', make_topk(7), 0),
    ('Random-to-Optimal', predict_random_to_optimal, 0),
    ('Copy Others', predict_copy_others, 0),
    ('Contradict Others', predict_contradict, 0),
]

for name, fn, n_params in baselines:
    print(f'Running {name}...')
    results = run_baseline(records, fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    metrics = extract_metrics(corrs)
    metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in ll.values()]))
    metrics['n_params'] = n_params
    all_models.append((name, metrics, corrs, results))
    print(f"  combo_r={metrics['combo_r']:.4f}, marg_r={metrics['marg_r']:.4f}")

Running Bayesian Walk...
  params: tau_prior=5.7100, tau_softmax=6.3695, epsilon=0.2000, epsilon_switch=0.3507
  combo_r=0.5407, marg_r=0.6269
Running Bayesian Threshold...
  params: tau_prior=5.6279, tau_softmax=3.8580, epsilon=0.1742, delta=3.3333
  combo_r=0.4744, marg_r=0.6055
Running Bayesian-Value...
  params: tau_prior=17.5678, tau_softmax=3.4170, epsilon=0.2000
  combo_r=0.4022, marg_r=0.4679
Running Random Walk...
  params: eps=0.2570
  combo_r=0.4510, marg_r=0.5275
Running Random...
  combo_r=0.1694, marg_r=nan
Running Optimal...
  combo_r=0.3097, marg_r=0.5025
Running Top-7...
  combo_r=0.3568, marg_r=0.4549
Running Random-to-Optimal...
  combo_r=0.2652, marg_r=0.4596
Running Copy Others...
  combo_r=0.0556, marg_r=0.5275
Running Contradict Others...
  combo_r=0.0965, marg_r=-0.5479


/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:126: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(combo_m, combo_h)
/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(marg_m, marg_h)
/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:143: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(global_marg_m, global_marg_h)
/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


## 6. Aggregate Summary Table

In [8]:
summary_rows = []
for name, metrics, corrs, results in all_models:
    summary_rows.append({
        'Model': name,
        'combo_r': metrics['combo_r'],
        'marg_r': metrics['marg_r'],
        'mean_ll': metrics.get('mean_ll', float('nan')),
        'params': metrics.get('n_params', '?'),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df = summary_df.sort_values('combo_r', ascending=False)

print('=== Aggregate Model Comparison (aggregate-tuned params) ===')
print(summary_df.to_string(float_format='{:.4f}'.format))
print(f'\nBest combo_r: {summary_df.combo_r.idxmax()} ({summary_df.combo_r.max():.4f})')
print(f'Best marg_r:  {summary_df.marg_r.idxmax()} ({summary_df.marg_r.max():.4f})')

summary_df.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; background-color: #90EE90')


=== Aggregate Model Comparison (aggregate-tuned params) ===
                    combo_r  marg_r  mean_ll  params
Model                                               
Bayesian Walk        0.5407  0.6269  -2.9646       4
Bayesian Threshold   0.4744  0.6055 -21.9843       4
Random Walk          0.4510  0.5275  -2.8763       1
Bayesian-Value       0.4022  0.4679  -4.4884       3
Top-7                0.3568  0.4549 -20.2795       0
Optimal              0.3097  0.5025 -34.8566       0
Random-to-Optimal    0.2652  0.4596 -11.6562       0
Random               0.1694     NaN  -3.2958       0
Contradict Others    0.0965 -0.5479  -4.2468       0
Copy Others          0.0556  0.5275 -30.9872       0

Best combo_r: Bayesian Walk (0.5407)
Best marg_r:  Bayesian Walk (0.6269)


,combo_r,marg_r,mean_ll,params
Model,,,,
Bayesian Walk,0.5407,0.6269,-2.9646,4.0000
Bayesian Threshold,0.4744,0.6055,-21.9843,4.0000
Random Walk,0.4510,0.5275,-2.8763,1.0000
Bayesian-Value,0.4022,0.4679,-4.4884,3.0000
Top-7,0.3568,0.4549,-20.2795,0.0000
Optimal,0.3097,0.5025,-34.8566,0.0000
Random-to-Optimal,0.2652,0.4596,-11.6562,0.0000
Random,0.1694,nan,-3.2958,0.0000
Contradict Others,0.0965,-0.5479,-4.2468,0.0000


## 7. Per-Environment Comparison

In [9]:
env_ids = sorted(set(r['env_id'] for r in records))
short_names = {
    'Bayesian Walk': 'BWalk',
    'Bayesian Threshold': 'BThresh',
    'Bayesian-Value': 'BValue',
    'Random Walk': 'RWalk',
    'Random': 'Rand',
    'Optimal': 'Opt',
    'Top-7': 'Top7',
    'Random-to-Optimal': 'R->Opt',
    'Copy Others': 'Copy',
    'Contradict Others': 'Contra',
}

env_rows = []
for env_id in env_ids:
    row = {'env_id': env_id}
    for name, metrics, corrs, results in all_models:
        sn = short_names.get(name, name)
        env_corr = corrs.get(env_id, {})
        row[sn] = env_corr.get('marginal', {}).get('r', float('nan'))
    env_rows.append(row)

global_row = {'env_id': 'GLOBAL'}
for name, metrics, corrs, results in all_models:
    sn = short_names.get(name, name)
    global_row[sn] = metrics['marg_r']
env_rows.append(global_row)

env_df = pd.DataFrame(env_rows).set_index('env_id')
print('=== Per-Environment marg_r ===')
print(env_df.to_string(float_format='{:.3f}'.format))

env_df.style.format('{:.3f}').highlight_max(axis=1, props='font-weight: bold; background-color: #90EE90')


=== Per-Environment marg_r ===
                 BWalk  BThresh  BValue  RWalk  Rand   Opt   Top7  R->Opt  Copy  Contra
env_id                                                                                 
114_222_222_MFF  0.398    0.726   0.696  0.036   NaN 0.793  0.727   0.696 0.036  -0.119
222_222_222_FFF  0.838    0.881   0.776  0.579   NaN 0.763  0.805   0.480 0.579  -0.579
411_141_114_FFM  0.795    0.668   0.849  0.788   NaN 0.676  0.866   0.653 0.788  -0.844
411_141_114_FTF  0.770    0.919   0.858  0.674   NaN 0.739  0.705   0.906 0.674  -0.814
411_141_114_FTM  0.799    0.527   0.099  0.790   NaN 0.000 -0.005  -0.003 0.790  -0.761
411_222_222_FMM  0.388    0.275  -0.165  0.420   NaN 0.155  0.317   0.082 0.420  -0.469
GLOBAL           0.627    0.606   0.468  0.527   NaN 0.502  0.455   0.460 0.527  -0.548


,BWalk,BThresh,BValue,RWalk,Rand,Opt,Top7,R->Opt,Copy,Contra
env_id,,,,,,,,,,
114_222_222_MFF,0.398,0.726,0.696,0.036,nan,0.793,0.727,0.696,0.036,-0.119
222_222_222_FFF,0.838,0.881,0.776,0.579,nan,0.763,0.805,0.480,0.579,-0.579
411_141_114_FFM,0.795,0.668,0.849,0.788,nan,0.676,0.866,0.653,0.788,-0.844
411_141_114_FTF,0.770,0.919,0.858,0.674,nan,0.739,0.705,0.906,0.674,-0.814
411_141_114_FTM,0.799,0.527,0.099,0.790,nan,0.000,-0.005,-0.003,0.790,-0.761
411_222_222_FMM,0.388,0.275,-0.165,0.420,nan,0.155,0.317,0.082,0.420,-0.469
GLOBAL,0.627,0.606,0.468,0.527,nan,0.502,0.455,0.460,0.527,-0.548


## 8. Switch-Stage Analysis

Performance of switch-stage-tuned models on switch stages (stages where at least one player changed role).

In [10]:
sw_tuned_models = []

# --- Bayesian Walk (switch-tuned) ---
p = bw_params['switch_stage_tuned']
print('Running Bayesian Walk (switch-tuned)...')
print(f"  params: tau_prior={p['tau_prior']:.4f}, tau_softmax={p['tau_softmax']:.4f}, epsilon={p['epsilon']:.4f}, epsilon_switch={p['epsilon_switch']:.4f}")
bw_sw_results = run_predictions(records, make_bayesian_walk(
    p['tau_prior'], p['tau_softmax'], p['epsilon'], p['epsilon_switch']))
bw_sw_filtered = filter_switch_stages(bw_sw_results)
bw_sw_corrs = compute_pearson(bw_sw_filtered)
bw_sw_ll = compute_log_likelihood(bw_sw_filtered)
bw_sw_metrics = extract_metrics(bw_sw_corrs)
bw_sw_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in bw_sw_ll.values()]))
sw_tuned_models.append(('Bayesian Walk (sw-tuned)', bw_sw_metrics))
print(f"  combo_r={bw_sw_metrics['combo_r']:.4f}, marg_r={bw_sw_metrics['marg_r']:.4f}")

# --- Bayesian Threshold (switch-tuned) ---
p = bt_params['switch_stage_tuned']
if 'bayesian_thresh' in p:
    p = p['bayesian_thresh']
print('Running Bayesian Threshold (switch-tuned)...')
print(f"  params: tau_prior={p['tau_prior']:.4f}, tau_softmax={p['tau_softmax']:.4f}, epsilon={p['epsilon']:.4f}, delta={p['delta']:.4f}")
bt_sw_results = run_predictions(records, make_bayesian_thresh(
    p['tau_prior'], p['tau_softmax'], p['epsilon'], p['delta']))
bt_sw_filtered = filter_switch_stages(bt_sw_results)
bt_sw_corrs = compute_pearson(bt_sw_filtered)
bt_sw_ll = compute_log_likelihood(bt_sw_filtered)
bt_sw_metrics = extract_metrics(bt_sw_corrs)
bt_sw_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in bt_sw_ll.values()]))
sw_tuned_models.append(('Bayesian Threshold (sw-tuned)', bt_sw_metrics))
print(f"  combo_r={bt_sw_metrics['combo_r']:.4f}, marg_r={bt_sw_metrics['marg_r']:.4f}")

# --- Bayesian-Value (switch-tuned) ---
p = bv_params['switch_stage_tuned']
print('Running Bayesian-Value (switch-tuned)...')
print(f"  params: tau_prior={p['tau_prior']:.4f}, tau_softmax={p['tau_softmax']:.4f}, epsilon={p['epsilon']:.4f}")
bv_sw_results = oms.run_all_predictions(records, tau_prior=p['tau_prior'],
                                         tau_softmax=p['tau_softmax'], epsilon=p['epsilon'])
bv_sw_filtered = filter_switch_stages(bv_sw_results)
bv_sw_corrs = compute_pearson(bv_sw_filtered)
bv_sw_ll = compute_log_likelihood(bv_sw_filtered)
bv_sw_metrics = extract_metrics(bv_sw_corrs)
bv_sw_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in bv_sw_ll.values()]))
sw_tuned_models.append(('Bayesian-Value (sw-tuned)', bv_sw_metrics))
print(f"  combo_r={bv_sw_metrics['combo_r']:.4f}, marg_r={bv_sw_metrics['marg_r']:.4f}")

# --- Random Walk (switch-tuned) ---
p = rw_params['switch_stage_tuned']
print('Running Random Walk (switch-tuned)...')
print(f"  params: eps={p['eps']:.4f}")
rw_sw_results = run_predictions(records, make_random_walk(p['eps']))
rw_sw_filtered = filter_switch_stages(rw_sw_results)
rw_sw_corrs = compute_pearson(rw_sw_filtered)
rw_sw_ll = compute_log_likelihood(rw_sw_filtered)
rw_sw_metrics = extract_metrics(rw_sw_corrs)
rw_sw_metrics['mean_ll'] = float(np.mean([v['mean_ll'] for v in rw_sw_ll.values()]))
sw_tuned_models.append(('Random Walk (sw-tuned)', rw_sw_metrics))
print(f"  combo_r={rw_sw_metrics['combo_r']:.4f}, marg_r={rw_sw_metrics['marg_r']:.4f}")

# --- Baselines on switch stages (no tuning needed) ---
baseline_names = {'Random', 'Optimal', 'Top-7', 'Random-to-Optimal', 'Copy Others', 'Contradict Others'}
for name, metrics, corrs, results in all_models:
    if name in baseline_names:
        sw_results = filter_switch_stages(results)
        if sw_results:
            sw_corrs = compute_pearson(sw_results)
            sw_ll = compute_log_likelihood(sw_results)
            sw_m = extract_metrics(sw_corrs)
            sw_m['mean_ll'] = float(np.mean([v['mean_ll'] for v in sw_ll.values()]))
        else:
            sw_m = {'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
        sw_tuned_models.append((name, sw_m))

# --- Summary ---
print('\n=== Switch-Stage Performance (switch-tuned params + baselines) ===')
sw_tuned_rows = [{'Model': n, 'combo_r': m['combo_r'], 'marg_r': m['marg_r'],
                   'mean_ll': m.get('mean_ll', float('nan'))} for n, m in sw_tuned_models]
sw_tuned_df = pd.DataFrame(sw_tuned_rows).set_index('Model')
sw_tuned_df = sw_tuned_df.sort_values('combo_r', ascending=False)
print(sw_tuned_df.to_string(float_format='{:.4f}'.format))
print(f'\nBest combo_r: {sw_tuned_df.combo_r.idxmax()} ({sw_tuned_df.combo_r.max():.4f})')

sw_tuned_df.style.format('{:.4f}').highlight_max(axis=0, props='font-weight: bold; background-color: #90EE90')


Running Bayesian Walk (switch-tuned)...
  params: tau_prior=10.0000, tau_softmax=0.1711, epsilon=0.0698, epsilon_switch=0.8152
  combo_r=0.2971, marg_r=0.3867
Running Bayesian Threshold (switch-tuned)...
  params: tau_prior=20.0000, tau_softmax=0.1000, epsilon=0.0750, delta=0.0000
  combo_r=0.2493, marg_r=0.3903
Running Bayesian-Value (switch-tuned)...
  params: tau_prior=0.1000, tau_softmax=0.1000, epsilon=0.0596
  combo_r=0.2865, marg_r=0.3873
Running Random Walk (switch-tuned)...
  params: eps=0.4958
  combo_r=0.2053, marg_r=0.3588

=== Switch-Stage Performance (switch-tuned params + baselines) ===
                               combo_r  marg_r  mean_ll
Model                                                  
Bayesian Walk (sw-tuned)        0.2971  0.3867 -19.0843
Bayesian-Value (sw-tuned)       0.2865  0.3873 -25.8343
Bayesian Threshold (sw-tuned)   0.2493  0.3903 -39.9226
Random-to-Optimal               0.2292  0.4274 -13.5518
Optimal                         0.2254  0.4003 -36.4602

/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:126: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(combo_m, combo_h)
/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(marg_m, marg_h)
/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:143: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(global_marg_m, global_marg_h)
/Users/bhavyesh/Desktop/bayesian-role-specialization/analysis/shared/evaluation.py:129: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(marg_m, marg_h)


,combo_r,marg_r,mean_ll
Model,,,
Bayesian Walk (sw-tuned),0.2971,0.3867,-19.0843
Bayesian-Value (sw-tuned),0.2865,0.3873,-25.8343
Bayesian Threshold (sw-tuned),0.2493,0.3903,-39.9226
Random-to-Optimal,0.2292,0.4274,-13.5518
Optimal,0.2254,0.4003,-36.4602
Random Walk (sw-tuned),0.2053,0.3588,-3.1889
Top-7,0.1821,0.3574,-24.4467
Random,0.1204,nan,-3.2958
Copy Others,0.0430,0.3588,-42.4680
